# Module 1 - Data Collection & Preparation

## Overview
This notebook covers the data preparation phase of the Driver Monitoring AI project.
We worked with 6 real datasets and generated 2 synthetic datasets for use in later modules.

## Real Datasets Cleaned

| Dataset | Rows | Columns | Notebook |
|---|---|---|---|
| Delivery Truck Trips | 3,451 | 23 | DeliveryTruckTripsData_cleaned.ipynb |
| Driver Behavior & Route Anomaly | 120,000 | 22 | DriverBehaviorandRouteAnomalyDatasetCleaned.ipynb |
| Driving Behavior (Train) | 3,644 | 8 | DrivingBehavior2Datasets.ipynb |
| Driving Behavior (Test) | 3,084 | 8 | DrivingBehavior2Datasets.ipynb |
| Traffic Violations | 1,499,945 | 24 | TrafficViolations.ipynb |
| Transportation & Logistics | 703 | 11 | TransportationAndLogistics.ipynb |

## Synthetic Datasets Generated
- **telemetry_samples.csv** → 1,000 records of driver telematics with full fields including speed, throttle, brake, GPS, accelerometer, event codes, and ratings
- **feedback.csv** → 500 passenger feedback entries with sentiment labels

In [1]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

# Sample feedback templates
positive_feedback = [
    "Great driver, very professional and on time!",
    "Smooth ride, driver was courteous and friendly.",
    "Excellent service, would definitely ride again.",
    "Driver followed all traffic rules, felt very safe.",
    "Very comfortable trip, driver was polite and helpful.",
    "Arrived early, clean vehicle and great attitude.",
    "Best ride experience, highly recommend this driver.",
    "Driver was calm and drove very smoothly throughout.",
]

neutral_feedback = [
    "Trip was okay, nothing special to mention.",
    "Average experience, driver was quiet the whole time.",
    "Decent ride, but took a slightly longer route.",
    "It was fine, arrived at destination without issues.",
    "Normal trip, driver did not interact much.",
    "Acceptable service, but vehicle could be cleaner.",
]

negative_feedback = [
    "Driver was speeding and it felt very unsafe.",
    "Very rude driver, did not follow the planned route.",
    "Hard braking multiple times, very uncomfortable ride.",
    "Driver was late and did not apologize at all.",
    "Terrible experience, driver was on the phone while driving.",
    "Driver ignored traffic signals, felt dangerous.",
    "Very rough driving, lots of sudden braking and acceleration.",
    "Driver took wrong turns and got us lost.",
]

N = 500
trip_ids = [f"T{i:04d}" for i in range(N)]
sentiments = np.random.choice(["positive", "neutral", "negative"], N, p=[0.5, 0.3, 0.2])

feedback_texts = []
for s in sentiments:
    if s == "positive":
        feedback_texts.append(random.choice(positive_feedback))
    elif s == "neutral":
        feedback_texts.append(random.choice(neutral_feedback))
    else:
        feedback_texts.append(random.choice(negative_feedback))

feedback_df = pd.DataFrame({
    "trip_id": trip_ids,
    "feedback_text": feedback_texts,
    "sentiment_label": sentiments,
    "timestamp": pd.date_range(start="2024-01-01", periods=N, freq="3h")
})

feedback_df.to_csv("../data/feedback.csv", index=False)
print(f"Feedback dataset created: {feedback_df.shape}")
feedback_df.head()

Feedback dataset created: (500, 4)


,trip_id,feedback_text,sentiment_label,timestamp
0,T0000,"Smooth ride, driver was courteous and friendly.",positive,2024-01-01 00:00:00
1,T0001,Driver was speeding and it felt very unsafe.,negative,2024-01-01 03:00:00
2,T0002,"Acceptable service, but vehicle could be cleaner.",neutral,2024-01-01 06:00:00
3,T0003,"Decent ride, but took a slightly longer route.",neutral,2024-01-01 09:00:00
4,T0004,"Driver followed all traffic rules, felt very s...",positive,2024-01-01 12:00:00


In [3]:
# Synthetic Telematics - Full Version
N = 1000

ip_ids = [f"IP{i:04d}" for i in range(N)]
driver_ids = np.random.choice([f"D{d:03d}" for d in range(50)], N)
trip_ids = [f"T{i:04d}" for i in range(N)]
timestamps = pd.date_range(start="2024-01-01", periods=N, freq="30min")
speeds = np.clip(np.random.normal(40, 12, N), 0, 140)
throttle = np.clip(np.random.normal(0.5, 0.2, N), 0, 1).round(2)
brake = np.clip(np.random.normal(0.2, 0.1, N), 0, 1).round(2)
steering_angle = np.random.uniform(-45, 45, N).round(2)
gps_lat = np.random.uniform(14.0, 14.9, N).round(6)
gps_lon = np.random.uniform(120.9, 121.9, N).round(6)
accel_x = np.random.normal(0, 0.5, N).round(3)
accel_y = np.random.normal(0, 0.5, N).round(3)
accel_z = np.random.normal(9.8, 0.2, N).round(3)
hard_brake = (np.random.rand(N) < 0.05).astype(int)
overspeed = (speeds > 80).astype(int)
event_code = np.where(hard_brake == 1, "hard_brake",
             np.where(overspeed == 1, "overspeed", "normal"))
trip_duration_sec = np.random.randint(300, 7200, N)
distance_km = (speeds * trip_duration_sec / 3600).round(2)
ratings = (5 - (speeds/50) - hard_brake*1.2 +
           np.random.normal(0, 0.5, N)).clip(1, 5).round().astype(int)
complaints_count = np.where(ratings <= 2, np.random.randint(1, 5, N), 0)

telemetry_df = pd.DataFrame({
    "ip_id": ip_ids,
    "driver_id": driver_ids,
    "trip_id": trip_ids,
    "timestamp": timestamps,
    "speed": speeds.round(2),
    "throttle": throttle,
    "brake": brake,
    "steering_angle": steering_angle,
    "gps_lat": gps_lat,
    "gps_lon": gps_lon,
    "accel_x": accel_x,
    "accel_y": accel_y,
    "accel_z": accel_z,
    "event_code": event_code,
    "hard_brake": hard_brake,
    "trip_duration_sec": trip_duration_sec,
    "distance_km": distance_km,
    "rating": ratings,
    "complaints_count": complaints_count
})

telemetry_df.to_csv("../data/telemetry_samples.csv", index=False)
print(f"Telemetry dataset created: {telemetry_df.shape}")
telemetry_df.head()

Telemetry dataset created: (1000, 19)


,ip_id,driver_id,trip_id,timestamp,speed,throttle,brake,steering_angle,gps_lat,gps_lon,accel_x,accel_y,accel_z,event_code,hard_brake,trip_duration_sec,distance_km,rating,complaints_count
0,IP0000,D031,T0000,2024-01-01 00:00:00,53.17,0.71,0.06,-9.63,14.065671,120.918158,0.278,-0.649,10.035,normal,0,6843,101.06,4,0
1,IP0001,D002,T0001,2024-01-01 00:30:00,21.21,0.43,0.15,41.56,14.739013,121.622461,-0.113,-0.731,10.027,normal,0,6689,39.41,5,0
2,IP0002,D046,T0002,2024-01-01 01:00:00,3.91,0.33,0.17,-7.58,14.443732,121.735536,-0.215,0.505,9.886,normal,0,4507,4.89,5,0
3,IP0003,D034,T0003,2024-01-01 01:30:00,46.85,0.69,0.16,9.10,14.762105,121.299142,1.004,-0.757,9.902,normal,0,5589,72.74,5,0
4,IP0004,D013,T0004,2024-01-01 02:00:00,42.92,0.43,0.14,18.88,14.540666,121.347942,-0.736,0.137,9.885,normal,0,6018,71.75,5,0
